# Phase 1 — Dataset Setup

This notebook is the **exploratory runner** for Phase 1 of the Heart Disease ML project.

The goal is to confirm that the UCI Heart Disease Cleveland, Hungarian, and Swiss splits can be loaded reproducibly, have aligned columns, and can support the project design:

- Cleveland = training/development source domain
- Hungarian = external test domain
- Swiss = external test domain

This notebook intentionally **does not write files**. It only displays checks inside the notebook. Persistent outputs such as cleaned CSVs, tables, figures, and model artifacts should be generated later by Python scripts under `src/` or `scripts/`.

## Prompt

Think about what makes a good clinical ML dataset: binary labels, external validation potential, and no DUA delays. Confirm that the Heart Disease UCI Cleveland dataset is available and that the Hungarian and Swiss splits are publicly available as separate files.

Plan: download all three splits, verify column alignment and label encoding across splits, then document the train/validation split strategy.

Check: this is Day 1, so no prior steps are required. Confirm the setup is reproducible.

Execute: download and load the data in a notebook, print shapes and label distributions for each split.

Evaluate: verify that the Cleveland split has about 300 rows and roughly balanced classes, and that Hungarian/Swiss can serve as true external test sets without data leakage.

## Imports

Only standard analysis imports are used here. The notebook does not save external artifacts.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Dataset configuration

The processed UCI files use the same 14 clinical columns. The original target has values 0–4, where 0 means no disease and 1–4 indicate disease presence. For this project, the target is collapsed to binary:

- `0` = no heart disease
- `1` = heart disease present

In [2]:
COLUMN_NAMES = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target",
]

DATA_URLS = {
    "cleveland": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
    "hungarian": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data",
    "swiss": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.switzerland.data",
}

SOURCE_DOMAIN = "cleveland"
EXTERNAL_DOMAINS = ["hungarian", "swiss"]

## Loading functions

These functions are written inside the notebook for Day 1 exploration. After this phase is finalized, the stable version should be moved to `src/data.py`.

In [7]:
def load_uci_heart_split(url: str) -> pd.DataFrame:
    """Load one processed UCI Heart Disease split directly from URL.

    Missing values are encoded as '?' in the raw files and are converted to NaN.
    The target is binarized: 0 remains 0, while 1-4 become 1.
    """
    df = pd.read_csv(
        url,
        header=None,
        names=COLUMN_NAMES,
        na_values="?",
    )

    df["target_original"] = df["target"]
    df["target"] = (df["target"] > 0).astype(int)
    return df


def load_all_splits(urls: dict[str, str]) -> dict[str, pd.DataFrame]:
    """Load all configured UCI Heart Disease splits."""
    return {name: load_uci_heart_split(url) for name, url in urls.items()}

In [9]:
datasets = load_all_splits(DATA_URLS)

for split_name, df in datasets.items():
    print(f"{split_name}: shape = {df.shape}")

cleveland: shape = (303, 15)
hungarian: shape = (294, 15)
swiss: shape = (123, 15)


## Column alignment check

All three splits must have the same clinical columns. The extra `target_original` column is kept only for verification and should not be used as a model feature.

In [10]:
expected_columns = COLUMN_NAMES + ["target_original"]

for split_name, df in datasets.items():
    aligned = list(df.columns) == expected_columns
    print(f"{split_name}: column alignment = {aligned}")
    if not aligned:
        print(df.columns.tolist())

cleveland: column alignment = True
hungarian: column alignment = True
swiss: column alignment = True


## Label distribution check

The project uses binary classification. This cell checks both the original target values and the collapsed binary target.

In [11]:
for split_name, df in datasets.items():
    print(f"\n=== {split_name.upper()} ===")
    print("Original target distribution:")
    print(df["target_original"].value_counts(dropna=False).sort_index())

    print("\nBinary target distribution:")
    counts = df["target"].value_counts(dropna=False).sort_index()
    proportions = df["target"].value_counts(normalize=True, dropna=False).sort_index()
    display(pd.DataFrame({"count": counts, "proportion": proportions}))


=== CLEVELAND ===
Original target distribution:
target_original
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

Binary target distribution:


,count,proportion
target,,
0,164,0.541254
1,139,0.458746



=== HUNGARIAN ===
Original target distribution:
target_original
0    188
1    106
Name: count, dtype: int64

Binary target distribution:


,count,proportion
target,,
0,188,0.639456
1,106,0.360544



=== SWISS ===
Original target distribution:
target_original
0     8
1    48
2    32
3    30
4     5
Name: count, dtype: int64

Binary target distribution:


,count,proportion
target,,
0,8,0.065041
1,115,0.934959


## Missing-value overview

This quick check identifies which columns require imputation. The imputer will later be fitted only on the Cleveland training split.

In [12]:
for split_name, df in datasets.items():
    print(f"\n=== {split_name.upper()} missing values ===")
    missing = df.isna().sum()
    display(missing[missing > 0].sort_values(ascending=False).to_frame("missing_count"))


=== CLEVELAND missing values ===


,missing_count
ca,4
thal,2



=== HUNGARIAN missing values ===


,missing_count
ca,291
thal,266
slope,190
chol,23
fbs,8
trestbps,1
restecg,1
thalach,1
exang,1



=== SWISS missing values ===


,missing_count
ca,118
fbs,75
thal,52
slope,17
oldpeak,6
trestbps,2
restecg,1
thalach,1
exang,1


## Train/validation split strategy

The leakage-safe strategy is:

1. Use **Cleveland only** for model development.
2. Split Cleveland into train and validation/holdout sets using stratification.
3. Fit preprocessing, feature engineering parameters, hyperparameter optimization, and calibration only using Cleveland train/validation data.
4. Keep Hungarian and Swiss untouched until final external evaluation.

Hungarian and Swiss represent external domains/sites and should not be used for fitting imputers, scalers, feature selectors, calibrators, Optuna objective decisions, thresholds, or model parameters.

In [13]:
cleveland = datasets[SOURCE_DOMAIN]

X_cleveland = cleveland.drop(columns=["target", "target_original"])
y_cleveland = cleveland["target"]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleveland,
    y_cleveland,
    test_size=0.20,
    random_state=42,
    stratify=y_cleveland,
)

print("Cleveland train shape:", X_train.shape)
print("Cleveland holdout shape:", X_holdout.shape)
print("Train positive rate:", round(y_train.mean(), 3))
print("Holdout positive rate:", round(y_holdout.mean(), 3))

Cleveland train shape: (242, 13)
Cleveland holdout shape: (61, 13)
Train positive rate: 0.459
Holdout positive rate: 0.459


## External test split check

This cell confirms that Hungarian and Swiss are kept separate from the Cleveland development split.

In [14]:
for split_name in EXTERNAL_DOMAINS:
    df = datasets[split_name]
    X_external = df.drop(columns=["target", "target_original"])
    y_external = df["target"]
    print(f"{split_name}: external X shape = {X_external.shape}, positive rate = {y_external.mean():.3f}")

hungarian: external X shape = (294, 13), positive rate = 0.361
swiss: external X shape = (123, 13), positive rate = 0.935


## Phase 1 evaluation notes

Fill this section after running the notebook.

Expected conclusions:

- Cleveland has approximately 303 rows and can be used as the source development split.
- Hungarian and Swiss are available as separate processed files and can be treated as external test domains.
- The target can be safely collapsed from the original 0–4 scale into binary 0/1.
- Missing values exist and must be handled by an imputer fitted only on Cleveland train data.
- No preprocessing or model-selection decision should use Hungarian or Swiss data.